### Extracting

In [53]:
!pip install pdfplumber

In [54]:
import pdfplumber
import re

In [55]:
keep_as_is= {'لا', 'في', 'من', 'أو', 'إذا', 'على', 'إلى',
              'عن', 'مع', 'ما', 'إن', 'أن', 'لم', 'لن',
              'ثم', 'حتى', 'إلا', 'أما', 'كما', 'ولا', 'فلا'}


In [56]:
def normalize_arabic_numbers(text):
    arabic_nums = "٠١٢٣٤٥٦٧٨٩"
    western_nums = "0123456789"
    return text.translate(str.maketrans(arabic_nums, western_nums))

In [57]:
def fix_arabic_only(text):
    def reverse_arabic_word(match):
        word = match.group(0)
        if word in keep_as_is:
            return word
        return word[::-1]
    return re.sub(r'[\u0600-\u06FF]+', reverse_arabic_word, text)

In [58]:
def fix_arabic_word_order(text):
    lines = text.split("\n")
    fixed_lines = []
    for line in lines:
        arabic_words = re.findall(r'[\u0600-\u06FF]+', line)
        if arabic_words:
            words = line.split()
            fixed_lines.append(" ".join(words[::-1]))
        else:
            fixed_lines.append(line)
    return "\n".join(fixed_lines)

In [59]:
def clean_english(text):
    text = re.sub(r'[\(\)]\d+[\(\)]', '', text)
    text = re.sub(r'[\(\)]\s*[\(\)]', '', text)
    text = re.sub(r'\s[\(\)]\s*$', '', text)
    text = re.sub(r'^\s*[\(\)]\s', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [60]:
def fix_arabic(text):
    text = normalize_arabic_numbers(text)
    text = fix_arabic_only(text)
    text = fix_arabic_word_order(text)
    text = re.sub(r'[\(\)]\d+[\(\)]', '', text)
    text = re.sub(r'[\(\)]\s*[\(\)]', '', text)
    text = re.sub(r'\s[\(\)]\s*$', '', text)
    text = re.sub(r'^\s*[\(\)]\s', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [61]:
def is_arabic_token(word):
    return bool(re.search(r'[\u0600-\u06FF\u0660-\u0669]', word))

In [62]:
def page_to_lines(page):
    words = page.extract_words()
    if not words:
        return []
    words = sorted(words, key=lambda w: w['top'])
    tolerance = 6
    lines, current = [], [words[0]]
    for w in words[1:]:
        avg_top = sum(x['top'] for x in current) / len(current)
        if abs(w['top'] - avg_top) <= tolerance:
            current.append(w)
        else:
            lines.append(current)
            current = [w]
    lines.append(current)
    return lines

In [63]:
def extract_page(page):
    mid_x = page.width / 2
    lines = page_to_lines(page)
    english_lines = []
    arabic_lines  = []

    for line in lines:
        left_words   = [w for w in line if w['x0'] < mid_x]
        right_words  = [w for w in line if w['x0'] >= mid_x]

        left_english = [w for w in left_words  if not is_arabic_token(w['text'])]
        left_arabic  = [w for w in left_words  if is_arabic_token(w['text'])]
        right_arabic = [w for w in right_words if is_arabic_token(w['text'])]
        right_english= [w for w in right_words if not is_arabic_token(w['text'])]

        all_english = sorted(left_english + right_english, key=lambda w: w['x0'])
        all_arabic  = sorted(left_arabic  + right_arabic,  key=lambda w: w['x0'])

        if all_english:
            line_text = " ".join(w['text'] for w in all_english)
            cleaned = clean_english(line_text)
            if cleaned:
                english_lines.append(cleaned)

        if all_arabic:
            arabic_lines.append(fix_arabic(" ".join(w['text'] for w in all_arabic)))

    return "\n".join(english_lines), "\n".join(arabic_lines)

In [64]:
with pdfplumber.open("/content/egyptian_laws.pdf") as pdf:
    all_english, all_arabic = [], []
    for page in pdf.pages:
        eng, ara = extract_page(page)
        all_english.append(eng)
        all_arabic.append(ara)

In [65]:
english_text = "\n".join(all_english)
arabic_text  = "\n".join(all_arabic)

In [66]:
print("English:\n", english_text)
print("\nArabic:\n",  arabic_text)

English:
 SECTION I
Laws and their Applications
1. Laws and Rights
Article 1
Provisions of laws govern all matters to which these
provisions apply in letter or spirit.
In the absence of a provision of a law that is applicable,
the Judge will decide according to custom and in the
absence of custom in accordance with the principles of
Moslem Law.
In the absence of such principles, the Judge will apply the
principles of natural justice and the rules of equity.
Article 2
A provision of a law can only be repealed by a
subsequent law expressly providing for such repeal, or
containing a provision inconsistent with a provision of
the former law or regulating anew a matter previously
regulated by a former law.
Article 3
Periods of limitation will be calculated according to the
Gregorian calendar, unless expressly provided otherwise
by a law.
Article 4
A person legitimately exercises his rights is not
responsible for prejudice resulting thereby.
Article 5
The exercise of a right is considered un

### Jsonify

In [67]:
import re
import json

In [68]:
roman = r'(?:I{1,3}|IV|VI{0,3}|IX|XI{0,3}|XIV|XV?I{0,3})'

In [69]:
def parse_english(lines):
    articles = {}
    book = chapter = section = subsection = num = None
    body = []

    for line in lines:
        line = line.strip()
        if not line:
            continue

        if re.match(rf'^BOOK\s+{roman}', line, re.IGNORECASE):
            if num:
                articles[num] = {'book': book, 'chapter': chapter, 'section': section, 'subsection': subsection, 'text': ' '.join(body)}
            book, chapter, section, subsection, num, body = line, None, None, None, None, []

        elif re.match(rf'^CHAPTER\s+{roman}', line, re.IGNORECASE):
            if num:
                articles[num] = {'book': book, 'chapter': chapter, 'section': section, 'subsection': subsection, 'text': ' '.join(body)}
            chapter, section, subsection, num, body = line, None, None, None, []

        elif re.match(rf'^SECTION\s+{roman}', line, re.IGNORECASE):
            if num:
                articles[num] = {'book': book, 'chapter': chapter, 'section': section, 'subsection': subsection, 'text': ' '.join(body)}
            section, subsection, num, body = line, None, None, []

        elif re.match(r'^\d+[\.\-]\s+\S', line):
            subsection = line

        elif m := re.match(r'^Article\s+(\d+)\s*[\(\)]*\s*$', line, re.IGNORECASE):
            if num:
                articles[num] = {'book': book, 'chapter': chapter, 'section': section, 'subsection': subsection, 'text': ' '.join(body)}
            num, body = int(m.group(1)), []

        else:
            if num:
                body.append(line)

    if num:
        articles[num] = {'book': book, 'chapter': chapter, 'section': section, 'subsection': subsection, 'text': ' '.join(body)}

    return articles

In [70]:
def parse_arabic(lines):
    articles = {}
    num = None
    body = []

    for line in lines:
        line = line.strip()
        if not line:
            continue

        if m := re.match(r'^مادة\s*\(?\s*(\d+)\s*\)?\s*$', line):
            if num:
                articles[num] = {'text': ' '.join(body)}
            num, body = int(m.group(1)), []

        else:
            if num:
                body.append(line)

    if num:
        articles[num] = {'text': ' '.join(body)}

    return articles

In [71]:
def merge_articles(en, ar):
    articles = []
    for num in sorted(set(en) | set(ar)):
        e = en.get(num, {})
        a = ar.get(num, {})
        articles.append({
            'article_number': num,
            'book':           e.get('book'),
            'chapter':        e.get('chapter'),
            'section':        e.get('section'),
            'subsection':     e.get('subsection'),
            'english_text':   e.get('text', ''),
            'arabic_text':    a.get('text', '')
        })
    return articles

In [72]:
en = parse_english(english_text.split('\n'))
ar = parse_arabic(arabic_text.split('\n'))
articles = merge_articles(en, ar)

In [73]:
print(f"Total articles: {len(articles)}")

Total articles: 1093


In [74]:
print(json.dumps(articles[0], ensure_ascii=False, indent=2))

{
  "article_number": 1,
  "book": null,
  "chapter": null,
  "section": "SECTION I",
  "subsection": "1. Laws and Rights",
  "english_text": "Provisions of laws govern all matters to which these provisions apply in letter or spirit. In the absence of a provision of a law that is applicable, the Judge will decide according to custom and in the absence of custom in accordance with the principles of Moslem Law. In the absence of such principles, the Judge will apply the principles of natural justice and the rules of equity.",
  "arabic_text": "تسرى النصوص التشريعية على جميع المسائل التي تتناولها هذه النصوص في لفظها أو في .فحواها فإذا لم يوجد نص تشريعي يمكن تطبيقه، حكم القاضي بمقتضى العرف، فإذا لم يوجد، فبمقتضى مبادئ الشريعة اإلسالمية، فإذا لم يوجد فبمقتضى مبادئ القانون الطبيعي وقواعد .العدالة"
}


In [75]:
with open('/content/articles.json', 'w', encoding='utf-8') as f:
    json.dump(articles, f, ensure_ascii=False, indent=2)

### Chunking and its Justification

In [76]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('intfloat/multilingual-e5-large')
# embedding model (multilingual for ar and en)  , loaded now aashan nshof el max tokens to see if our data will fit in one chunk or will need chunking

In [77]:
lengths = []
for a in articles:
    combined = f"passage: SECTION: {a['section']} | SUB: {a['subsection']} | EN: {a['english_text']} | AR: {a['arabic_text']}"  # will embed both the english and arabic together for better retrieval quality
    tokens = tokenizer(combined, return_tensors='pt')
    lengths.append(tokens['input_ids'].shape[1])
    if (tokens['input_ids'].shape[1] ) > 512 :
      print(f'article number: {a['article_number']}' + ' '+ a['english_text'])

Token indices sequence length is longer than the specified maximum sequence length for this model (549 > 512). Running this sequence through the model will result in indexing errors


article number: 238 If the act by the debtor is for valuable consideration it can only be held invalid as against the creditor, if made by the debtor with the intent to defraud and if the other party to the contract was aware of the fraud. It suffices for the act to be considered fraudulent, if the debtor knew, at the time that it was effected, that he was insolvent: the other party is deemed to have had knowledge of the fraud of the debtor, if he was aware of the debtor's state of insolvency. If, however, the act entered into by the debtor was gratuitous, it is not valid as against the creditor, even if the transferee acted in good faith and it is established that the debtor did not commit fraud. If a transferee disposes of property transmitted to him by a debtor for valuable consideration, a creditor can only claim avoidance of the act by the debtor if it was made for valuable consideration and if the second as well as the first transferee both knew of the debtor's fraud, and when th

In [78]:
import numpy as np
print(f"mean: {np.mean(lengths):.1f}")
print(f"max: {max(lengths)}")
print(f"min: {min(lengths)}")
print(f"% under 512: {sum(1 for l in lengths if l < 512) / len(lengths) * 100:.1f}%")
print(f"number over 512: {sum(1 for l in lengths if l > 512)}")

mean: 189.8
max: 1017
min: 46
% under 512: 99.4%
number over 512: 7


Justification for the chunking strategy

We measured the token length of every article before deciding how to chunk.

Results:
- Mean tokens per article : 189.8
- Max tokens              : 1017
- Min tokens              : 46
- Articles under 512      : 1086 / 1093  (99.4%)
- Articles over 512       : 7    / 1093  (0.6%)

Our model (multilingual-e5-large) has a 512-token limit.
Since 99.4% of articles fit within that limit, we keep them as-is — one article = one chunk. Splitting them would break the legal meaning
for no reason, since each article is already a complete, self-contained rule.

For the 7 articles that are too long, we split at sentence boundaries (at "." or ";") rather than cutting at an arbitrary token count.
Each part keeps the article number and section in its text so that
even a partial match still tells you which article it came from.

In [79]:
def build_passage(article, part_num=None, part_text=None):
    identity = f"Article {article['article_number']}"
    if article['section']:    identity += f" | {article['section']}"
    if article['subsection']: identity += f" | {article['subsection']}"
    if part_num:              identity += f" (Part {part_num})"

    en = part_text if part_text else article['english_text']
    ar = article['arabic_text']
    return f"passage: {identity} | EN: {en} | AR: {ar}"

In [80]:
def count_tokens(text):
    return len(tokenizer(text)['input_ids'])

In [81]:
def split_into_parts(text, is_arabic=False):
    pattern = r'(?<=[.;،؛])\s+' if is_arabic else r'(?<=[.;])\s+'
    return re.split(pattern, text.strip())


In [82]:
def chunk_articles(articles, max_tokens=512):
    chunks = []
    over_limit = []

    for article in articles:
        if count_tokens(build_passage(article)) <= max_tokens:
            chunks.append({**article, 'chunk_id': str(article['article_number']), 'is_split': False, 'passage': build_passage(article)})
            continue

        over_limit.append(article['article_number'])
        en_sentences = split_into_parts(article['english_text'])
        ar_sentences = split_into_parts(article['arabic_text'], is_arabic=True)

        diff = len(en_sentences) - len(ar_sentences)
        if diff > 0: ar_sentences += [''] * diff
        else: en_sentences += [''] * -diff

        current_en, current_ar, part_num = [], [], 1
        for en, ar in zip(en_sentences, ar_sentences):
            test = {**article, 'english_text': ' '.join(current_en + [en]), 'arabic_text': ' '.join(current_ar + [ar])}
            if count_tokens(build_passage(test, part_num)) > max_tokens and current_en:
                part = {**article, 'english_text': ' '.join(current_en), 'arabic_text': ' '.join(current_ar)}
                chunks.append({**part, 'chunk_id': f"{article['article_number']}_part{part_num}", 'is_split': True, 'passage': build_passage(part, part_num)})
                current_en, current_ar, part_num = [en], [ar], part_num + 1
            else:
                current_en.append(en)
                current_ar.append(ar)

        if current_en:
            part = {**article, 'english_text': ' '.join(current_en), 'arabic_text': ' '.join(current_ar)}
            chunks.append({**part, 'chunk_id': f"{article['article_number']}_part{part_num}", 'is_split': True, 'passage': build_passage(part, part_num)})

    print(f"Articles split: {over_limit}")
    print(f"Total chunks: {len(chunks)} from {len(articles)} articles")
    return chunks

In [83]:
chunks = chunk_articles(articles)

Articles split: [238, 502, 563, 1063, 1065, 1089, 1143]
Total chunks: 1101 from 1093 articles


In [84]:
chunks

[{'article_number': 1,
  'book': None,
  'chapter': None,
  'section': 'SECTION I',
  'subsection': '1. Laws and Rights',
  'english_text': 'Provisions of laws govern all matters to which these provisions apply in letter or spirit. In the absence of a provision of a law that is applicable, the Judge will decide according to custom and in the absence of custom in accordance with the principles of Moslem Law. In the absence of such principles, the Judge will apply the principles of natural justice and the rules of equity.',
  'arabic_text': 'تسرى النصوص التشريعية على جميع المسائل التي تتناولها هذه النصوص في لفظها أو في .فحواها فإذا لم يوجد نص تشريعي يمكن تطبيقه، حكم القاضي بمقتضى العرف، فإذا لم يوجد، فبمقتضى مبادئ الشريعة اإلسالمية، فإذا لم يوجد فبمقتضى مبادئ القانون الطبيعي وقواعد .العدالة',
  'chunk_id': '1',
  'is_split': False,
  'passage': 'passage: Article 1 | SECTION I | 1. Laws and Rights | EN: Provisions of laws govern all matters to which these provisions apply in letter or spi

### Embeddings+ Vector DB

In [85]:
!pip install qdrant-client

In [86]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer

In [87]:
import os
from google.colab import userdata
os.environ["QDRANT_URL"] = userdata.get('QDRANT_URL')
os.environ["QDRANT_API_KEY"] = userdata.get('QDRANT_API_KEY')

In [88]:
model  = SentenceTransformer('intfloat/multilingual-e5-small')

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [89]:
passages= [c['passage'] for c in chunks]
embeddings = model.encode(passages, batch_size=32, show_progress_bar=True, normalize_embeddings=True)

Batches:   0%|          | 0/35 [00:00<?, ?it/s]

In [90]:
client = QdrantClient(
    url=os.environ.get("QDRANT_URL"),
    api_key=os.environ.get("QDRANT_API_KEY")
    )

In [91]:
existing = [c.name for c in client.get_collections().collections]
if "egyptian_civil_code" in existing:
    client.delete_collection("egyptian_civil_code")

In [92]:
client.create_collection(
    collection_name="egyptian_civil_code",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)


True

In [93]:
points = [
    PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload={
            'article_number':c['article_number'],
            'chunk_id':c['chunk_id'],
            'is_split':c['is_split'],
            'book':c['book'],
            'chapter':c['chapter'],
            'section':c['section'],
            'subsection':c['subsection'],
            'english_text':c['english_text'],
            'arabic_text':c['arabic_text'],
        }
    )
    for i, c in enumerate(chunks)
]

In [94]:
batch_size = 100
for i in range(0, len(points), batch_size):
    client.upsert(collection_name="egyptian_civil_code", points=points[i:i+batch_size])
    print(f"Uploaded {min(i+batch_size, len(points))}/{len(points)}")

Uploaded 100/1101
Uploaded 200/1101
Uploaded 300/1101
Uploaded 400/1101
Uploaded 500/1101
Uploaded 600/1101
Uploaded 700/1101
Uploaded 800/1101
Uploaded 900/1101
Uploaded 1000/1101
Uploaded 1100/1101
Uploaded 1101/1101


### Retrieval Trials

In [95]:
def search(query, top_k=3):
    query_embedding = model.encode(f"query: {query}", normalize_embeddings=True)
    results = client.query_points(
        collection_name="egyptian_civil_code",
        query=query_embedding.tolist(),
        limit=top_k
    )
    return results.points

In [96]:
def answer(query, top_k=3):
    results = search(query, top_k)

    for r in results:
        print(f"Article {r.payload['article_number']} (score: {r.score:.3f})")
        print(f"English: {r.payload['english_text'][:200]}...")
        print(f"Arabic: {r.payload['arabic_text']}...")
        print("-" * 50)

In [97]:
answer("Someone performed a service for me but they didn't know I had posted a reward for it online. Am I legally obligated to pay them once they find out?")

Article 220 (score: 0.845)
English: A formal summons to the debtor will not be necessary in the following cases: a) if the performance of the obligation becomes impossible or without interest by an act of the debtor; b) if the object of...
Arabic: لا ضرورة العذار المدين فى الحاالت :اآلتية أ إذا اصبح تنفيذ اإللتزام غير ممكن أو كير مجد بفعل .المدين ب إذا كان محل اإللتزام تعويضا ترتب على عمل غير .مشروع ج إذا كان محل اإللتزام رد شئ يعلم المدين انه مسروق أو شئ تسلمه دون حق وهو عالم .بذلك د إذا صرح المدين كتابة انه لا يريد القيام .بإلتزامه...
--------------------------------------------------
Article 162 (score: 0.843)
English: A person who makes a promise to the public of a reward in exchange for a specified service is bound to pay the reward to the person who performs the service, even if he acted without thought of the pr...
Arabic: 1 من وجه للجمهور وعدا بجائزة يعطيها عن عمل معين التزم بإعطاء الجائزة لمن قام بهذا العمل ولو قام به دون نظر إلى الوعد بالجائزة أو دون علم .بها 2) وإذا لم يعين 

In [98]:
answer("ما هي مبادئ القانون الطبيعي؟")

Article 1 (score: 0.813)
English: Provisions of laws govern all matters to which these provisions apply in letter or spirit. In the absence of a provision of a law that is applicable, the Judge will decide according to custom and in t...
Arabic: تسرى النصوص التشريعية على جميع المسائل التي تتناولها هذه النصوص في لفظها أو في .فحواها فإذا لم يوجد نص تشريعي يمكن تطبيقه، حكم القاضي بمقتضى العرف، فإذا لم يوجد، فبمقتضى مبادئ الشريعة اإلسالمية، فإذا لم يوجد فبمقتضى مبادئ القانون الطبيعي وقواعد .العدالة...
--------------------------------------------------
Article 200 (score: 0.805)
English: The judge shall decide, in the absence of any provision of the law, whether a natural obligation exists. There cannot ever be a natural obligation that is contrary to public policy....
Arabic: يقدر القاضى ، عند عدم النص ، ما إذا كان هناك إلتزام .طبيعى وفى كل حال لا يجوز أن يقوم إلتزام طبيعى يخالف النظام .العام...
--------------------------------------------------
Article 86 (score: 0.801)
English: Rights in